In [ ]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#     "numpy",
#     "pyarrow",
# ]
# ///

In [ ]:
keys = [
    "GSM2219497_Cell_1",
    "GSM2219498_Cell_2",
    "GSM2219499_Cell_3",
    "GSM2219500_Cell_4",
    "GSM2219501_Cell_5",
    "GSM2219502_Cell_6",
    "GSM2219503_Cell_7",
    "GSM2219504_Cell_8",
]

In [ ]:
import numpy as np
import pyarrow as pa
from pathlib import Path

def convert_to_arrow(filepath):
    atom_lines = []
    with open(filepath, 'r') as file:
        print("Hi")
        x_arr = []
        y_arr = []
        z_arr = []
        for line in file:
            # print(line)
            line_annot = line[0:6]
            if line_annot == "HETATM" or line_annot == "ATOM":
                atom_lines.append(line)
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                x_arr.append(x)
                y_arr.append(y)
                z_arr.append(z)

        print(len(atom_lines))
        x_nparr = np.array(x_arr)
        y_nparr = np.array(y_arr)
        z_nparr = np.array(z_arr)
        table = pa.Table.from_arrays([x_nparr, y_nparr, z_nparr], ["x", "y", "z"])

        out_folder = Path("out")
        out_folder.mkdir(parents=True, exist_ok=True)
        outfile_path = "out/" + key + ".arrow"
        with pa.OSFile(outfile_path, 'wb') as sink:
            with pa.ipc.RecordBatchFileWriter(sink, table.schema) as writer:
                writer.write_table(table)

In [ ]:
basepath = "GSE80280_RAW"
postfix = "_genome_structure_model.pdb"

for key in keys:
    filepath = f"{basepath}/{key}{postfix}"
    print(filepath)
    convert_to_arrow(filepath)
    print("Done.")